# Notebook 2 - Collaborative Filtering
Collaborative Filtering is a recommendation technique that identifies patterns in user behavior rather than looking at item content. The core idea is that users who agreed on ratings in the past are likely to agree in the future. If two users both loved the same set of movies, we can recommend movies that one user liked to the other.

SVD (Singular Value Decomposition) is a matrix factorization technique that compresses the large sparse user-movie matrix into smaller hidden representations called latent factors. These latent factors capture hidden patterns such as a user's preference for action movies or a movie's appeal to a certain audience type — without us ever explicitly defining these characteristics.

---

This notebook implements SVD based collaborative filtering on the MovieLens 1M dataset. The key steps include:

- Loading and formatting train and test data for the Surprise library
- Training a baseline SVD model with default parameters
- Evaluating model performance using RMSE and MAE
- Tuning hyperparameters using GridSearchCV with 3-fold cross validation
- Retraining the model with the best found parameters
- Generating top-N personalized movie recommendations for users
- Comparing recommendations against user's actual watch history
- Saving the trained model for use in later phases

Final Model Performance:
- RMSE: 0.9372
- MAE:  0.7440

The SVD model successfully learns latent user and movie factors to predict ratings and generate personalized recommendations.

In [1]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, GridSearchCV
from surprise import accuracy
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [2]:
# load data
train = pd.read_csv("OneDrive/Documents/Recommendation System/train.csv")
test = pd.read_csv("OneDrive/Documents/Recommendation System/test.csv")
movies = pd.read_csv("OneDrive/Documents/Recommendation System/movies.csv")

print("Train size:", train.shape)
print("Test size:", test.shape)
print(train.head())

Train size: (800167, 11)
Test size: (200042, 11)
   user_id  movie_id  rating            timestamp year_month  \
0     6040       858       4  2000-04-25 23:05:32    2000-04   
1     6040       593       5  2000-04-25 23:05:54    2000-04   
2     6040      2384       4  2000-04-25 23:05:54    2000-04   
3     6040      1961       4  2000-04-25 23:06:17    2000-04   
4     6040      2019       5  2000-04-25 23:06:17    2000-04   

                                               title              genres  \
0                              Godfather, The (1972)  Action|Crime|Drama   
1                   Silence of the Lambs, The (1991)      Drama|Thriller   
2                       Babe: Pig in the City (1998)   Children's|Comedy   
3                                    Rain Man (1988)               Drama   
4  Seven Samurai (The Magnificent Seven) (Shichin...        Action|Drama   

  gender  age  occupation zip_code  
0      M   25           6    11106  
1      M   25           6    11106 

## Prepare Data for Surprise

In [3]:
# prepare data for surprise
# surprise has its own data format
# Reader is used to tell it our ratings go from 1 to 5
reader = Reader(rating_scale=(1, 5))

# load train data into surprise format
train_data = Dataset.load_from_df(
    train[['user_id', 'movie_id', 'rating']],
    reader
)

# build the full trainset
trainset = train_data.build_full_trainset()

print("Number of users in trainset:", trainset.n_users)
print("Number of items in trainset:", trainset.n_items)
print("Number of ratings in trainset:", trainset.n_ratings)

Number of users in trainset: 5400
Number of items in trainset: 3662
Number of ratings in trainset: 800167


## Prepare Test Set for Surprise

In [4]:
# convert test data to surprise format for evaluation
testset = list(zip(
    test['user_id'].values,
    test['movie_id'].values,
    test['rating'].values
))

print("Number of ratings in testset:", len(testset))
print("Sample testset entry:", testset[0])

Number of ratings in testset: 200042
Sample testset entry: (1875, 1721, 4)


## Train the SVD Model

In [5]:
# initialize SVD with default parameter first
svd_model = SVD(
    n_factors=100,   # Number of latent factors
    n_epochs=20,     # Number of training iterations
    lr_all=0.005,    # Learning rate
    reg_all=0.02,    # Regularization to prevent overfitting
    random_state=42  # Ensures reproducibility
)

# train the model
svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


## Evaluate on Test Set

In [6]:
# make predictions on test set
predictions = svd_model.test(testset)

# calculate RMSE and MAE
# rmse - measures the average difference between predicted and actual ratings.
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\nRMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

RMSE: 0.9395
MAE:  0.7419

RMSE: 0.9395
MAE: 0.7419


## Hyperparameter Tuning with GridSearch

In [7]:
param_grid = {
    'n_factors': [50, 100],
    'n_epochs': [20, 30],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.1]
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=1)
gs.fit(train_data)

print("Best RMSE:", gs.best_score['rmse'])
print("Best RMSE Parameters:", gs.best_params['rmse'])
print("\nBest MAE:", gs.best_score['mae'])
print("Best MAE Parameters:", gs.best_params['mae'])

Best RMSE: 0.87927797565666
Best RMSE Parameters: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

Best MAE: 0.6968443112584571
Best MAE Parameters: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


## Retrain with Best Parameter

In [8]:
best_params = gs.best_params['rmse']

svd_best = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)

svd_best.fit(trainset)

# evaluate again
best_predictions = svd_best.test(testset)
print("Tuned RMSE:", accuracy.rmse(best_predictions))
print("Tuned MAE:", accuracy.mae(best_predictions))

RMSE: 0.9372
Tuned RMSE: 0.9371969187201932
MAE:  0.7440
Tuned MAE: 0.743971927785446


## Generate Top-N Recommendations for a User
This function generates actual recommendations. For a given user it finds all movies they haven't rated yet, predicts a rating for each one using our trained SVD model, sorts by predicted rating, and returns the top N. This is the core of what a recommendation system does — predict how much a user would enjoy something they haven't seen yet.

In [12]:
# Find most active users in train set
active_users = train['user_id'].value_counts().head(10)
print("Most active users in train set:")
print(active_users)

# Pick most active user automatically
test_user = active_users.index[0]
print(f"\nSelected User ID: {test_user}")

Most active users in train set:
user_id
1680    1849
889     1518
4169    1440
4277    1407
3618    1344
1941    1305
1150    1298
5795    1272
4344    1271
4510    1240
Name: count, dtype: int64

Selected User ID: 1680


In [13]:
# user history function
def get_user_history(user_id, train_df, movies_df, n=10):
    user_ratings = train_df[train_df['user_id'] == user_id].copy()
    
    if 'title' not in user_ratings.columns:
        user_ratings = user_ratings.merge(
            movies_df[['movie_id', 'title', 'genres']],
            on='movie_id',
            how='left'
        )
    
    return user_ratings[['title', 'genres', 'rating']].sort_values(
        'rating', ascending=False
    ).head(n)

print(f"Top rated movies by User {test_user}:")
print(get_user_history(test_user, train, movies))

Top rated movies by User 1680:
                                         title  \
676972                   Goofy Movie, A (1995)   
630913                       L.A. Story (1991)   
630673                      Coming Home (1978)   
630759           War of the Worlds, The (1953)   
630791        Kentucky Fried Movie, The (1977)   
630792  Sinbad and the Eye of the Tiger (1977)   
630854                   Eyes Wide Shut (1999)   
630855                  Sixteen Candles (1984)   
630911                             Cube (1997)   
630938     Star Trek: The Wrath of Khan (1982)   

                                     genres  rating  
676972  Animation|Children's|Comedy|Romance       5  
630913                       Comedy|Romance       5  
630673                            Drama|War       5  
630759                    Action|Sci-Fi|War       5  
630791                               Comedy       5  
630792                     Action|Adventure       5  
630854                                Dr

In [14]:
# recommendation function
def get_top_n_recommendations(user_id, model, train_df, movies_df, n=10):
    all_movie_ids = movies_df['movie_id'].unique()
    
    # Use set for faster lookup
    rated_movies = set(train_df[train_df['user_id'] == user_id]['movie_id'].values)
    
    unrated_movies = [m for m in all_movie_ids if m not in rated_movies]
    
    predictions = [model.predict(user_id, movie_id) for movie_id in unrated_movies]
    
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    top_n = predictions[:n]
    
    results = pd.DataFrame({
        'movie_id': [p.iid for p in top_n],
        'predicted_rating': [round(p.est, 2) for p in top_n]
    })
    
    results = results.merge(movies_df[['movie_id', 'title', 'genres']], on='movie_id')
    return results

print(f"Top 10 recommendations for User {test_user}:")
recommendations = get_top_n_recommendations(
    user_id=test_user,
    model=svd_best,
    train_df=train,
    movies_df=movies,
    n=10
)
print(recommendations[['title', 'genres', 'predicted_rating']])

Top 10 recommendations for User 1680:
                                               title                genres  \
0                            Apple, The (Sib) (1998)                 Drama   
1                                    Lamerica (1994)                 Drama   
2                              Paths of Glory (1957)             Drama|War   
3                           Julien Donkey-Boy (1999)                 Drama   
4                                     Sanjuro (1962)      Action|Adventure   
5        Grand Illusion (Grande illusion, La) (1937)             Drama|War   
6                              Third Man, The (1949)      Mystery|Thriller   
7                                     Yojimbo (1961)  Comedy|Drama|Western   
8                            Hearts and Minds (1996)                 Drama   
9  Eternity and a Day (Mia eoniotita ke mia mera ...                 Drama   

   predicted_rating  
0              4.70  
1              4.65  
2              4.63  
3              

## Compare History vs Recommendation

In [15]:
print(f"===== User {test_user} Profile =====\n")
print("--- Movies Already Rated (Top 10) ---")
print(get_user_history(test_user, train, movies))
print(f"\n--- Top 10 Recommendations ---")
print(recommendations[['title', 'genres', 'predicted_rating']])

===== User 1680 Profile =====

--- Movies Already Rated (Top 10) ---
                                         title  \
676972                   Goofy Movie, A (1995)   
630913                       L.A. Story (1991)   
630673                      Coming Home (1978)   
630759           War of the Worlds, The (1953)   
630791        Kentucky Fried Movie, The (1977)   
630792  Sinbad and the Eye of the Tiger (1977)   
630854                   Eyes Wide Shut (1999)   
630855                  Sixteen Candles (1984)   
630911                             Cube (1997)   
630938     Star Trek: The Wrath of Khan (1982)   

                                     genres  rating  
676972  Animation|Children's|Comedy|Romance       5  
630913                       Comedy|Romance       5  
630673                            Drama|War       5  
630759                    Action|Sci-Fi|War       5  
630791                               Comedy       5  
630792                     Action|Adventure       5  
63

## Save Model

In [16]:
import os
import pickle

models_path = "OneDrive/Documents/Recommendation System/models"
os.makedirs(models_path, exist_ok=True)

with open(f"{models_path}/svd_model.pkl", 'wb') as f:
    pickle.dump(svd_best, f)

print("SVD model saved successfully!")

SVD model saved successfully!
